# Compute the volume-scaled CCSN birthrate

In [ ]:
import argparse
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import sys
from contextlib import redirect_stdout, redirect_stderr
import os
import logging
import seaborn as sns

import utilities.plot_settings
from mlpoppyns.simulator.config_simulator import cfg
import mlpoppyns.simulator.stellar_dynamics.coordinate_conversions as cc
from mlpoppyns.simulator.simulate_population_dyn import simulate_population

In [ ]:
SMALL_SIZE = 8
MEDIUM_SIZE = 10
BIGGER_SIZE = 18

plt.rc("font", size=SMALL_SIZE)  # controls default text sizes
plt.rc("axes", titlesize=BIGGER_SIZE)  # fontsize of the axes title
plt.rc("axes", labelsize=BIGGER_SIZE)  # fontsize of the x and y labels
plt.rc("xtick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("ytick", labelsize=MEDIUM_SIZE)  # fontsize of the tick labels
plt.rc("legend", fontsize=MEDIUM_SIZE)  # legend fontsize
plt.rc("figure", titlesize=BIGGER_SIZE)  # fontsize of the figure title

In [ ]:
# List of files to compare.
main_path = "/Users/celsapardoaraujo/Documents/ML-Poppyns/data/paper_results/pardo_etal_2026/birth_rate_dyn_sim"

file_paths = [
    f"{main_path}/birthrate_data_all_rmVV21_8_3_sun_saYMW17.npz",
    f"{main_path}/birthrate_data_all_8_sun_132_hc_spiralmodel_saYMW17.npz",
    f"{main_path}/birthrate_data_all_8_sun_spiralmodel_saYMW17.npz",
    f"{main_path}/birthrate_data_all8_3_edm_180_hc.npz",
    f"{main_path}/birthrate_data_all_VV21_8_3_edm_180_hc_NE2001.npz"
]

In [ ]:
data = np.load(file_paths[0])


# Access the arrays
birthrate_list = data["birthrate"]
stars_within_1kpc_list = data["within1kpc"]

# Create a mask for values where stars_within_1kpc_list <= 5
mask = stars_within_1kpc_list <= 10

# Apply the mask to birthrate_list
birthrate_list = birthrate_list[mask]

# If you want to also filter stars_within_1kpc_list accordingly:
stars_within_1kpc_list = stars_within_1kpc_list[mask]

In [ ]:
# Plot distributions
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

sns.histplot(birthrate_list,bins=np.arange(0, 40, 1), ax=axs[0], color="skyblue")
axs[0].set_xlabel("Birthrate Distribution")
axs[0].set_ylabel("Count")
axs[0].set_xlim(0, 40)  # <-- Limit x-axis to 0–100
#axs[0].set_xticks(np.arange(0, 100, 1))  # Show all integer ticks from 10 to 20
#axs[0].set_yscale("log")  # <-- Apply log scale to y-axis

sns.histplot(stars_within_1kpc_list, bins=np.arange(0, 40, 1), ax=axs[1], color="salmon")
axs[1].set_xlabel("Stars within 2 kpc")
axs[1].set_ylabel("Count")
axs[1].set_xlim(0, 15)  # <-- Limit x-axis to 0–100
axs[1].set_xticks(np.arange(0, 15, 2))  # Show all integer ticks from 10 to 20

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# Convert lists to numpy arrays
birthrate_arr = np.array(birthrate_list)
stars_within_1kpc_arr = np.array(stars_within_1kpc_list)

# Compute mean and std
birthrate_mean = np.mean(birthrate_arr)
birthrate_median = np.median(birthrate_arr)
birthrate_std = np.std(birthrate_arr)

stars_within_1kpc_mean = np.mean(stars_within_1kpc_arr)
stars_within_1kpc_std = np.std(stars_within_1kpc_arr)
stars_within_1kpc_median = np.median(stars_within_1kpc_arr)

median = np.median(birthrate_list)

# 68% central interval (16th to 84th percentiles)
p16 = np.percentile(birthrate_list, 16)
p84 = np.percentile(birthrate_list, 84)

# 95% central interval (2.5th to 97.5th percentiles)
p025 = np.percentile(birthrate_list, 2.5)
p975 = np.percentile(birthrate_list, 97.5)

# Append results
ci68_lower = p16
ci68_upper = p84 
ci95_lower = p025
ci95_upper = p975 

print(f"Birthrate: median and 1 sigma =  {ci68_lower,birthrate_median,ci68_upper}")
print(f"Birthrate: median and 3 sigma  =  {ci95_lower,birthrate_median,ci95_upper}")

print(f"Birthrate: mean = {birthrate_mean:.2f}, std = {birthrate_std:.2f}")
print(f"Birthrate: median = {birthrate_median:.2f}, std = {birthrate_std:.2f}")
print(f"Stars within 1 kpc: mean = {stars_within_1kpc_mean:.2f}, std = {stars_within_1kpc_std:.2f}")


In [ ]:
# List of files to compare
file_paths = [
    "/Users/celsapardoaraujo/Documents/data/magnetar-br/birth_rate_dyn_sim/birthrate_data_all_8_sun_132_hc_spiralmodel_saYMW17.npz",
    "/Users/celsapardoaraujo/Documents/data/magnetar-br/birth_rate_dyn_sim/birthrate_data_all_8_sun_132_hc_spiralmodel_saYMW17.npz",
    "/Users/celsapardoaraujo/Documents/data/magnetar-br/birth_rate_dyn_sim/birthrate_data_all_8_sun_132_hc_spiralmodel_saYMW17.npz"
]

# Labels for the datasets
dataset_labels = ["HC", "LC", "MC"]

# Store stats for all datasets
all_medians = []
all_stds = []

# Load and process each file
for file_path in file_paths:
    data = np.load(file_path)
    birthrate_list = data["birthrate"].tolist()
    
    medians = np.median(birthrate_list) 
    stds = np.std(birthrate_list) 
    
    all_medians.append(medians)
    all_stds.append(stds)



In [ ]:
# List of files to compare
file_paths = [
    "/Users/celsapardoaraujo/Documents/data/magnetar-br/birth_rate_dyn_sim/birthrate_data_all_rmVV21_8_3_sun_saYMW17.npz",
    "/Users/celsapardoaraujo/Documents/data/magnetar-br/birth_rate_dyn_sim/birthrate_data_all_8_sun_132_hc_spiralmodel_saYMW17.npz",
    "/Users/celsapardoaraujo/Documents/data/magnetar-br/birth_rate_dyn_sim/birthrate_data_all_8_sun_spiralmodel_saYMW17.npz",
    "/Users/celsapardoaraujo/Documents/data/magnetar-br/birth_rate_dyn_sim/birthrate_data_all8_3_edm_180_hc.npz",
    "/Users/celsapardoaraujo/Documents/data/magnetar-br/birth_rate_dyn_sim/birthrate_data_all_VV21_8_3_edm_180_hc_NE2001.npz"
]

# Dataset labels
dataset_labels = ["Spiral model Sun at 8.3, h_c 180 pc", "Spiral model Sun at 8 h_c, 132 pc", "Spiral model Sun at 8 h_c, 180 pc",'Electron density YAO2016','Electron density NE2001']

# Containers
all_medians = []
ci68_lower = []
ci68_upper = []
ci95_lower = []
ci95_upper = []

# Process each dataset
for file_path in file_paths:
    data = np.load(file_path)
    birthrate_list = data["birthrate"]

    # Flatten if needed
    if isinstance(birthrate_list[0], (list, np.ndarray)):
        birthrate_list = np.concatenate(birthrate_list)
    
    # Compute statistics
    median = np.median(birthrate_list)
    
    # 68% central interval (16th to 84th percentiles)
    p16 = np.percentile(birthrate_list, 16)
    p84 = np.percentile(birthrate_list, 84)
    
    # 95% central interval (2.5th to 97.5th percentiles)
    p025 = np.percentile(birthrate_list, 2.5)
    p975 = np.percentile(birthrate_list, 97.5)

    # Append results
    all_medians.append(median)
    ci68_lower.append(median - p16)
    ci68_upper.append(p84 - median)
    ci95_lower.append(median - p025)
    ci95_upper.append(p975 - median)

# Plotting
x = np.arange(len(dataset_labels))
plt.figure(figsize=(10, 6))

# 95% CI (wide, light)
plt.errorbar(
    x,
    all_medians,
    yerr=[ci95_lower, ci95_upper],
    fmt='o',
    capsize=6,
    ecolor='orange',
    color='blue',
    alpha=0.6,
    label='Median ± 95% CI'
)

# 68% CI (narrow, darker)
plt.errorbar(
    x,
    all_medians,
    yerr=[ci68_lower, ci68_upper],
    fmt='o',
    capsize=6,
    ecolor='green',
    color='blue',
    alpha=0.9,
    label='Median ± 68% CI'
)

# Final formatting
plt.xticks(x, dataset_labels, rotation=30, ha='right')
plt.ylabel("Birthrate Estimate")
#plt.title("Birthrate Estimates: Median ± 68% and 95% Quantile Intervals")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# List of files to compare
file_paths = [
    "/Users/celsapardoaraujo/Documents/data/magnetar-br/birth_rate_dyn_sim/birthrate_data_all_rmVV21_8_3_sun_saYMW17.npz",
    "/Users/celsapardoaraujo/Documents/data/magnetar-br/birth_rate_dyn_sim/birthrate_data_all_8_sun_132_hc_spiralmodel_saYMW17.npz",
    "/Users/celsapardoaraujo/Documents/data/magnetar-br/birth_rate_dyn_sim/birthrate_data_all_8_sun_spiralmodel_saYMW17.npz"
]

# Dataset labels
dataset_labels = ["Spiral model Sun at 8.3, h_c 180 pc", "Spiral model Sun at 8 h_c, 132 pc", "Spiral model Sun at 8 h_c, 180 pc"]
# Containers
all_medians = []
ci68_lower = []
ci68_upper = []
ci95_lower = []
ci95_upper = []

# Process each dataset
for file_path in file_paths:
    data = np.load(file_path)
    birthrate_list = data["birthrate"]

    # Flatten if needed
    if isinstance(birthrate_list[0], (list, np.ndarray)):
        birthrate_list = np.concatenate(birthrate_list)
    
    # Compute statistics
    median = np.median(birthrate_list)
    
    # 68% central interval (16th to 84th percentiles)
    p16 = np.percentile(birthrate_list, 16)
    p84 = np.percentile(birthrate_list, 84)
    
    # 95% central interval (2.5th to 97.5th percentiles)
    p025 = np.percentile(birthrate_list, 2.5)
    p975 = np.percentile(birthrate_list, 97.5)

    # Append results
    all_medians.append(median)
    ci68_lower.append(median - p16)
    ci68_upper.append(p84 - median)
    ci95_lower.append(median - p025)
    ci95_upper.append(p975 - median)

# Plotting
x = np.arange(len(dataset_labels))
plt.figure(figsize=(10, 6))

# 95% CI (wide, light)
plt.errorbar(
    x,
    all_medians,
    yerr=[ci95_lower, ci95_upper],
    fmt='o',
    capsize=6,
    ecolor='orange',
    color='blue',
    alpha=0.6,
    label='Median ± 95% CI'
)

# 68% CI (narrow, darker)
plt.errorbar(
    x,
    all_medians,
    yerr=[ci68_lower, ci68_upper],
    fmt='o',
    capsize=6,
    ecolor='green',
    color='blue',
    alpha=0.9,
    label='Median ± 68% CI'
)

# Final formatting
plt.xticks(x, dataset_labels, rotation=30, ha='right')
plt.ylabel("Birthrate Estimate")
#plt.title("Birthrate Estimates: Median ± 68% and 95% Quantile Intervals")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
